# Detailed variant statistics for VEPs
Here we get detailed variant statistics for variant groups of interest

In [1]:
# Make sure our package is installed (run once)
#!pip install -e /home/dataproc/repos/igvf-cvfg-biobank

Notes on running in AoU's DataProc: as of 2026-09-25, trying to use pandas with the `fastparquet` engine throws an error. Manually forcing an update of relevant packages resolves it:

In [2]:
# Run once in dataproc
#!pip install --upgrade numpy numba fastparquet pandas statsmodels

In the standard environment, `fastparquet` is not installed, but we use it to append to a parquet file:

In [3]:
# Run once in a standard env
#!pip install fastparquet

In [4]:
import os
import pandas as pd
from pathlib import Path
from functools import cache
import hail as hl
from cvfgaou import notation, hailtools
from tqdm.notebook import tqdm

In [5]:
# In standard env
#wgs_mt_path = "/home/jupyter/workspace/All of Us Controlled Tier v8/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"

In [6]:
# In Dataproc
wgs_mt_path = 'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/splitMT/hail.mt'

In [7]:
# Constants (mostly paths to things)

variant_table_path = 'variant_tables/annotated_filtered_VEP_calibrations.tsv.gz'
gene_cohorts_path = 'gene_cohorts'

google_project = os.getenv('GOOGLE_PROJECT')

vep_variant_counts_path = Path('variant_groups/vep-variant-counts_2026-10-10.parquet')

# Index column labels
index_cols=['Dataset/Predictor', 'Gene', 'Grouping/Calibration', 'Consequence', 'Group/Range']

In [8]:
# Load dataframe
variants_df = pd.read_table(variant_table_path, dtype=str)

In [9]:
variants_df

In [10]:
variants_df['POS'] = pd.to_numeric(variants_df['POS'], downcast='integer')

Now establish the sets of variants to test.

First we inspect the columns that will define variant sets:

In [11]:
variants_df[['consequence']].value_counts(dropna=False)

In [12]:
score_cols = variants_df.columns[variants_df.columns.str.endswith('score')]
score_cols

In [13]:
variants_df[score_cols] = variants_df[score_cols].apply(pd.to_numeric)

In [14]:
points_cols = variants_df.columns[variants_df.columns.str.startswith('points')]
points_cols

In [15]:
variants_df[points_cols] = variants_df[points_cols].apply(pd.to_numeric, downcast='integer')

In [16]:
# Limit to genes with case/control cohorts established
genes_with_cohorts = {
    gene for gene in variants_df['Gene'].unique()
    if Path(gene_cohorts_path, f'{gene} cohort.parquet').exists()
}

In [17]:
genes_with_cohorts

For each predictor, we examine the three calibration strategies for both missense only and all variants.

In [18]:
from cvfgaou.notation import LEQ_CHAR, GEQ_CHAR

predictors = ['AlphaMissense', 'MutPred2', 'REVEL']
calibrators = ['single_gene', 'domain_aggregate']

threshold_lists = { # Bergquist et al. thresholds 10.1016/j.gim.2025.101402
    'MutPred2': [
        (f'{LEQ_CHAR} -4', pd.Series.le, 0.010), # Benign Strong
        (f'{LEQ_CHAR} -3', pd.Series.le, 0.031), # Benign Moderate+
        (f'{LEQ_CHAR} -2', pd.Series.le, 0.197), # Benign Moderate
        (f'{LEQ_CHAR} -1', pd.Series.le, 0.391), # Benign Supporting
        (f'{GEQ_CHAR} +1', pd.Series.ge, 0.737), # Pathogenic Supporting
        (f'{GEQ_CHAR} +2', pd.Series.ge, 0.829), # Pathogenic Moderate
        (f'{GEQ_CHAR} +3', pd.Series.ge, 0.895), # Pathogenic Moderate+
        (f'{GEQ_CHAR} +4', pd.Series.ge, 0.932)  # Pathogenic Strong
    ],
    'AlphaMissense': [
        (f'{LEQ_CHAR} -3', pd.Series.le, 0.070), # Benign Moderate+
        (f'{LEQ_CHAR} -2', pd.Series.le, 0.099), # Benign Moderate
        (f'{LEQ_CHAR} -1', pd.Series.le, 0.169), # Benign Supporting
        (f'{GEQ_CHAR} +1', pd.Series.ge, 0.792), # Pathogenic Supporting
        (f'{GEQ_CHAR} +2', pd.Series.ge, 0.906), # Pathogenic Moderate
        (f'{GEQ_CHAR} +3', pd.Series.ge, 0.972), # Pathogenic Moderate+
        (f'{GEQ_CHAR} +4', pd.Series.ge, 0.990)  # Pathogenic Strong
    ],
    'REVEL': [
        (f'{GEQ_CHAR} +4', pd.Series.ge, 0.932), # Pathogenic strong
        (f'{GEQ_CHAR} +3', pd.Series.ge, 0.879), # -- modetate+
        (f'{GEQ_CHAR} +2', pd.Series.ge, 0.773), # -- moderate
        (f'{GEQ_CHAR} +1', pd.Series.ge, 0.644), # Pathogenic supporting
        (f'{LEQ_CHAR} -1', pd.Series.le, 0.290), # Benign supporting
        (f'{LEQ_CHAR} -2', pd.Series.le, 0.183), # -- moderate
        (f'{LEQ_CHAR} -3', pd.Series.le, 0.052), # -- moderate+
        (f'{LEQ_CHAR} -4', pd.Series.le, 0.016) # -- strong
    ]
}

In [19]:
# Genome-wide calibration groups
genome_wide_groups = {
    (predictor, gene, 'genome_wide', consequence, classification):
        df[comparator(df[f'{predictor}_score'], threshold) & consequence_selection]
    for gene, df in variants_df.groupby('Gene')
    for predictor, threshold_list in threshold_lists.items()
    for (classification, comparator, threshold) in threshold_list
    for consequence, consequence_selection in (
        ('All', True),
        ('Missense', ~df['consequence'].isna())
    )
    if gene in genes_with_cohorts
}

In [20]:
# Other calibration groups
advanced_calibration_groups = {
    (predictor, gene, calibration, consequence, classification):
        df[classification_selection & consequence_selection]
    for gene, df in variants_df.groupby('Gene')
    for predictor in predictors
    for calibration in calibrators
    for classification, classification_selection in [
        (f'{GEQ_CHAR} {points:+d}', pd.to_numeric(df[f'points_{predictor}_{calibration}']) >= points)
        for points in range(1,5)
    ] + [
        (f'{LEQ_CHAR} {-points:+d}', pd.to_numeric(df[f'points_{predictor}_{calibration}']) <= -points)
        for points in range(1,9)
    ]
    for consequence, consequence_selection in (
        ('All', True),
        ('Missense', ~df['consequence'].isna())
    )
    if gene in genes_with_cohorts
}

In [21]:
variant_groups = genome_wide_groups | advanced_calibration_groups

Next, count variants in variant groups

In [22]:
# Check for missing NT coordinates
for label, df in variant_groups.items():
    if df[['#CHROM', 'POS', 'REF', 'ALT']].isna().any(axis=None):
        display(label)
        display(df)

Function for retrieving cohort hail table efficiently

In [23]:
@cache
def get_cohort_ht(gene):
    cohort_df = pd.read_parquet(Path(gene_cohorts_path, f'{gene} cohort.parquet'))
    cohort_df['s'] = cohort_df.index.to_series().astype(str)
    return hl.Table.from_pandas(
        cohort_df,
        key='s'
    ).cache()

Next, get the counts for each variant group

In [24]:
hl.init(
    #default_reference='GRCh38', deprecated
    gcs_requester_pays_configuration=google_project
)

Run the counter functions

In [25]:
# WGS table

wgs_mt = hl.read_matrix_table(wgs_mt_path)

In [26]:
# Attempt to check if we are adding to an existing file
excluded_groups = set()
if vep_variant_counts_path.exists():
    carriers_df = pd.read_parquet(vep_variant_counts_path)
    excluded_groups = set(
        carriers_df.
        index.
        to_frame()[index_cols].
        drop_duplicates().
        itertuples(index=False, name=None)
    )
    display(f'Existing file found with {len(excluded_groups)} groups.')

for variant_group, df in tqdm(variant_groups.items()):
    if variant_group in excluded_groups:
        continue

    result = {
        'Variants in Group': df[['#CHROM', 'POS', 'REF', 'ALT']].drop_duplicates().shape[0]
    }
    
    if not df.empty:
        result |= hailtools.get_variant_counts(
            hailtools.get_filtered_mt(
                df[['#CHROM', 'POS', 'REF', 'ALT']], wgs_mt,
                contig_col='#CHROM',
                pos_col='POS',
                ref_col='REF',
                alt_col='ALT',
                reference_genome='GRCh38'
            ),
            get_cohort_ht(variant_group[1]) # Gene is the 2nd field
        )
    else:
        result |= {
            'Variants in WGS MT': None,
            'Variants in cohort': 0,
            'Case only variant count': 0,
            'Control only variant count': 0,
            'Overlap variant count': 0
        }
    
    result_df = pd.DataFrame.from_dict(
        {variant_group: result},
        orient='index'
    )
    result_df.index.set_names(index_cols, inplace=True)

    result_df.to_parquet(
        vep_variant_counts_path,
        engine='fastparquet',
        append=vep_variant_counts_path.exists()
    )